In [ ]:
import polars as pl
import matplotlib.pyplot as plt
import numpy as np

# Compute number of hospital stays by admission year
stays_by_year = (
    df
    .with_columns(
        pl.col("hosp_admission_datetime").dt.year().alias("admission_year")
    )
    .drop_nulls(["admission_year"])
    .group_by("admission_year")
    .agg(pl.len().alias("n_hospital_stays"))
    .sort("admission_year")
)

# If result is LazyFrame, collect it
if isinstance(stays_by_year, pl.LazyFrame):
    stays_by_year = stays_by_year.collect()

years = stays_by_year["admission_year"].to_numpy()
counts = stays_by_year["n_hospital_stays"].to_numpy()

plt.figure(figsize=(9, 5))

bars = plt.bar(
    years,
    counts,
    width=0.75,
    edgecolor="white",
    linewidth=1.1,
    alpha=0.85
)

# Add count labels above bars
for bar, count in zip(bars, counts):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height(),
        f"{count:,}",
        ha="center",
        va="bottom",
        fontsize=9
    )

plt.xlabel("Admission year", fontsize=12)
plt.ylabel("Number of hospital stays", fontsize=12)
plt.title("Number of hospital stays by admission year", fontsize=14, weight="bold")

plt.xticks(years, rotation=45)
plt.grid(axis="y", linestyle="--", alpha=0.3)

ax = plt.gca()
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.ylim(0, max(counts) * 1.15)

plt.tight_layout()

plt.savefig(
    "hospital_stays_by_admission_year.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
import polars as pl
import matplotlib.pyplot as plt
import numpy as np

# Number of hospital stays per patient
stays_per_patient = (
    df
    .group_by("patient_id")
    .agg(pl.col("hosp_id").n_unique().alias("n_hospital_stays"))
)

if isinstance(stays_per_patient, pl.LazyFrame):
    stays_per_patient = stays_per_patient.collect()

stay_counts = stays_per_patient["n_hospital_stays"].to_numpy()

mean_stays = np.mean(stay_counts)
median_stays = np.median(stay_counts)
max_stays = int(np.max(stay_counts))

# Count patients for each exact number of hospital stays
stay_count_dist = (
    stays_per_patient
    .group_by("n_hospital_stays")
    .agg(pl.len().alias("n_patients"))
    .sort("n_hospital_stays")
)

x = stay_count_dist["n_hospital_stays"].to_numpy()
y = stay_count_dist["n_patients"].to_numpy()

plt.figure(figsize=(10, 5))

bars = plt.bar(
    x,
    y,
    width=0.8,
    edgecolor="white",
    linewidth=1.1,
    alpha=0.85
)

# Add labels only for high bars to avoid clutter
for bar, count in zip(bars, y):
    if count >= max(y) * 0.08:
        plt.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height(),
            f"{count:,}",
            ha="center",
            va="bottom",
            fontsize=9
        )

plt.axvline(
    mean_stays,
    linestyle="--",
    linewidth=2,
    label=f"Mean = {mean_stays:.1f}"
)

plt.axvline(
    median_stays,
    linestyle="-",
    linewidth=2,
    label=f"Median = {median_stays:.1f}"
)

plt.xlabel("Number of hospital stays per patient", fontsize=12)
plt.ylabel("Number of patients", fontsize=12)
plt.title("Distribution of the number of hospital stays per patient", fontsize=14, weight="bold")

plt.grid(axis="y", linestyle="--", alpha=0.3)
plt.legend(frameon=False)

ax = plt.gca()
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

# Important: limit x-axis to the actual maximum value
plt.xlim(0, max_stays + 1)

plt.ylim(0, max(y) * 1.15)

plt.tight_layout()

plt.savefig(
    "number_of_hospital_stays_per_patient.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print(f"Maximum number of hospital stays per patient: {max_stays}")

In [ ]:
import polars as pl
import matplotlib.pyplot as plt
import numpy as np

stays_per_patient = (
    df
    .group_by("patient_id")
    .agg(pl.col("hosp_id").n_unique().alias("n_hospital_stays"))
)

if isinstance(stays_per_patient, pl.LazyFrame):
    stays_per_patient = stays_per_patient.collect()

max_display = 15

stays_per_patient_binned = stays_per_patient.with_columns(
    pl.when(pl.col("n_hospital_stays") >= max_display)
    .then(pl.lit(f">= {max_display}"))
    .otherwise(pl.col("n_hospital_stays").cast(pl.Utf8))
    .alias("stay_group")
)

stay_count_dist = (
    stays_per_patient_binned
    .group_by("stay_group")
    .agg(pl.len().alias("n_patients"))
)

# Manual ordering
order_labels = [str(i) for i in range(1, max_display)] + [f">= {max_display}"]

stay_count_dist = (
    stay_count_dist
    .with_columns(
        pl.col("stay_group")
        .replace({label: i for i, label in enumerate(order_labels)})
        .cast(pl.Int64)
        .alias("order")
    )
    .sort("order")
)

x_labels = stay_count_dist["stay_group"].to_list()
y = stay_count_dist["n_patients"].to_list()

plt.figure(figsize=(9, 5))

bars = plt.bar(
    x_labels,
    y,
    edgecolor="white",
    linewidth=1.1,
    alpha=0.85
)

for bar, count in zip(bars, y):
    if count >= max(y) * 0.05:
        plt.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height(),
            f"{count:,}",
            ha="center",
            va="bottom",
            fontsize=9
        )

plt.xlabel("Number of hospital stays per patient", fontsize=12)
plt.ylabel("Number of patients", fontsize=12)
plt.title("Distribution of hospital stays per patient", fontsize=14, weight="bold")

plt.grid(axis="y", linestyle="--", alpha=0.3)

ax = plt.gca()
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.ylim(0, max(y) * 1.15)

plt.tight_layout()

plt.savefig(
    "number_of_hospital_stays_per_patient_binned.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
import polars as pl
import matplotlib.pyplot as plt
import numpy as np

# If df is LazyFrame, collect only after aggregation
stays_per_patient = (
    df
    .group_by("patient_id")
    .agg(pl.col("hosp_id").n_unique().alias("n_hospital_stays"))
)

if isinstance(stays_per_patient, pl.LazyFrame):
    stays_per_patient = stays_per_patient.collect()

stay_counts = stays_per_patient["n_hospital_stays"].to_numpy()

mean_stays = np.mean(stay_counts)
median_stays = np.median(stay_counts)
max_stays = np.max(stay_counts)

# Discrete distribution: number of patients for each number of stays
stay_count_dist = (
    stays_per_patient
    .group_by("n_hospital_stays")
    .agg(pl.len().alias("n_patients"))
    .sort("n_hospital_stays")
)

x = stay_count_dist["n_hospital_stays"].to_numpy()
y = stay_count_dist["n_patients"].to_numpy()

plt.figure(figsize=(9, 5))

bars = plt.bar(
    x,
    y,
    width=0.75,
    edgecolor="white",
    linewidth=1.1,
    alpha=0.85
)

# Add labels only for larger bars to avoid clutter
for bar, count in zip(bars, y):
    if count >= max(y) * 0.05:
        plt.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height(),
            f"{count:,}",
            ha="center",
            va="bottom",
            fontsize=9
        )

plt.axvline(
    mean_stays,
    linestyle="--",
    linewidth=2,
    label=f"Mean = {mean_stays:.1f}"
)

plt.axvline(
    median_stays,
    linestyle="-",
    linewidth=2,
    label=f"Median = {median_stays:.1f}"
)

plt.xlabel("Number of hospital stays per patient", fontsize=12)
plt.ylabel("Number of patients", fontsize=12)
plt.title("Distribution of the number of hospital stays per patient", fontsize=14, weight="bold")

plt.grid(axis="y", linestyle="--", alpha=0.3)
plt.legend(frameon=False)

ax = plt.gca()
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.ylim(0, max(y) * 1.15)

plt.tight_layout()

plt.savefig(
    "number_of_hospital_stays_per_patient.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
import polars as pl
import matplotlib.pyplot as plt
import numpy as np

# If df_patient is LazyFrame, collect it
if isinstance(df_patient, pl.LazyFrame):
    df_patient_plot = df_patient.collect()
else:
    df_patient_plot = df_patient

# Add comorbidity_count if it does not already exist
if "comorbidity_count" not in df_patient_plot.columns:
    df_patient_plot = df_patient_plot.with_columns(
        pl.sum_horizontal([pl.col(c) for c in comorbidity_cols]).alias("comorbidity_count")
    )

count_alive = (
    df_patient_plot
    .filter(pl.col("death_status") == "Alive")
    .select("comorbidity_count")
    .drop_nulls()
    .get_column("comorbidity_count")
    .to_numpy()
)

count_dead = (
    df_patient_plot
    .filter(pl.col("death_status") == "Dead")
    .select("comorbidity_count")
    .drop_nulls()
    .get_column("comorbidity_count")
    .to_numpy()
)

data = [count_alive, count_dead]
labels = ["Alive", "Dead"]

plt.figure(figsize=(6.5, 4.8))

box = plt.boxplot(
    data,
    labels=labels,
    patch_artist=True,
    widths=0.55,
    showmeans=True,
    meanline=True,
    medianprops={"linewidth": 2},
    meanprops={"linewidth": 2, "linestyle": "--"},
    boxprops={"linewidth": 1.3},
    whiskerprops={"linewidth": 1.2},
    capprops={"linewidth": 1.2},
    flierprops={"marker": "o", "markersize": 3, "alpha": 0.30}
)

for patch in box["boxes"]:
    patch.set_alpha(0.75)

# Add n and mean above each group
for i, values in enumerate(data, start=1):
    plt.text(
        i,
        np.max(values) + 0.4,
        f"n = {len(values):,}\nMean = {np.mean(values):.2f}",
        ha="center",
        va="bottom",
        fontsize=10
    )

plt.xlabel("Death status", fontsize=12)
plt.ylabel("Number of comorbidities", fontsize=12)
plt.title("Number of comorbidities according to death status", fontsize=14, weight="bold")

plt.grid(axis="y", linestyle="--", alpha=0.3)

ax = plt.gca()
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

all_counts = np.concatenate(data)
plt.ylim(np.min(all_counts) - 0.5, np.max(all_counts) + 2)

plt.tight_layout()

plt.savefig(
    "comorbidity_count_by_death_status.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
import polars as pl
import matplotlib.pyplot as plt
import numpy as np

# If df_patient is LazyFrame, collect it
if isinstance(df_patient, pl.LazyFrame):
    df_patient_plot = df_patient.collect()
else:
    df_patient_plot = df_patient

# Add comorbidity count per patient
df_patient_plot = df_patient_plot.with_columns(
    pl.sum_horizontal([pl.col(c) for c in comorbidity_cols]).alias("comorbidity_count")
)

# Count patients for each number of comorbidities
comorbidity_count_dist = (
    df_patient_plot
    .group_by("comorbidity_count")
    .agg(pl.len().alias("n"))
    .with_columns(
        (pl.col("n") / pl.col("n").sum() * 100).alias("percent")
    )
    .sort("comorbidity_count")
)

x = comorbidity_count_dist["comorbidity_count"].to_list()
y = comorbidity_count_dist["n"].to_list()
p = comorbidity_count_dist["percent"].to_list()

plt.figure(figsize=(8, 5))

bars = plt.bar(
    x,
    y,
    width=0.75,
    edgecolor="white",
    linewidth=1.1,
    alpha=0.85
)

# Add count and percentage labels
for bar, count, percent in zip(bars, y, p):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height(),
        f"{count:,}\n({percent:.1f}%)",
        ha="center",
        va="bottom",
        fontsize=9
    )

plt.xlabel("Number of comorbidities per patient", fontsize=12)
plt.ylabel("Number of patients", fontsize=12)
plt.title("Distribution of the number of comorbidities per patient", fontsize=14, weight="bold")

plt.xticks(range(0, len(comorbidity_cols) + 1))
plt.grid(axis="y", linestyle="--", alpha=0.3)

ax = plt.gca()
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.ylim(0, max(y) * 1.18)

plt.tight_layout()

plt.savefig(
    "number_of_comorbidities_per_patient.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
import polars as pl
import matplotlib.pyplot as plt
import numpy as np

# If df_patient is LazyFrame, collect it
if isinstance(df_patient, pl.LazyFrame):
    df_patient_plot = df_patient.collect()
else:
    df_patient_plot = df_patient

# French labels for the report
comorbidity_labels = {
    "Cardiopathie ischémique": "Cardiopathie ischémique",
    "Fibrillation atriale": "Fibrillation atriale",
    "Insuffisance cardiaque chronique": "Insuffisance cardiaque chronique",
    "Pacemaker": "Pacemaker",
    "Pontage aorto-coronarien": "Pontage aorto-coronarien",
    "Insuffisance rénale chronique": "Insuffisance rénale chronique",
    "Antécédent d'AVC": "Antécédent d'AVC",
    "Cirrhose": "Cirrhose",
    "Cancer actif": "Cancer actif",
    "Immunodépression": "Immunodépression",
    "Dyslipidémie": "Dyslipidémie",
    "Diabète": "Diabète",
}

# Optional: French labels for death status
death_status_labels = {
    "Alive": "Vivant",
    "Dead": "Décédé",
}

# Compute prevalence by death status
comorbidity_by_death = []

for c in comorbidity_cols:
    tmp = (
        df_patient_plot
        .group_by("death_status")
        .agg(
            (pl.col(c).sum() / pl.len() * 100).alias("prevalence_percent")
        )
        .with_columns(pl.lit(c).alias("comorbidity"))
    )
    comorbidity_by_death.append(tmp)

comorbidity_by_death = pl.concat(comorbidity_by_death)

# Convert to pandas for easier plotting
comorbidity_by_death_pd = comorbidity_by_death.to_pandas()

pivot_comorb = (
    comorbidity_by_death_pd
    .pivot(
        index="comorbidity",
        columns="death_status",
        values="prevalence_percent"
    )
    .fillna(0)
)

# Keep French comorbidity labels
pivot_comorb.index = [
    comorbidity_labels.get(x, x) for x in pivot_comorb.index
]

# Keep only existing death-status columns in a stable order
death_order = [x for x in ["Alive", "Dead"] if x in pivot_comorb.columns]
pivot_comorb = pivot_comorb[death_order]

# Sort by prevalence among deceased patients if Dead exists, otherwise by first column
sort_col = "Dead" if "Dead" in pivot_comorb.columns else pivot_comorb.columns[0]
pivot_comorb = pivot_comorb.sort_values(by=sort_col)

# Rename columns to French for legend
pivot_comorb = pivot_comorb.rename(columns=death_status_labels)

# Plot
y_pos = np.arange(len(pivot_comorb.index))
bar_height = 0.38

plt.figure(figsize=(10, 7))

for i, status in enumerate(pivot_comorb.columns):
    values = pivot_comorb[status].values
    offset = (i - (len(pivot_comorb.columns) - 1) / 2) * bar_height

    bars = plt.barh(
        y_pos + offset,
        values,
        height=bar_height,
        label=status,
        edgecolor="white",
        linewidth=1.0,
        alpha=0.85
    )

    for bar, value in zip(bars, values):
        if value > 0:
            plt.text(
                value + 0.4,
                bar.get_y() + bar.get_height() / 2,
                f"{value:.1f}%",
                va="center",
                fontsize=9
            )

plt.yticks(y_pos, pivot_comorb.index, fontsize=10)
plt.xlabel("Prévalence parmi les patients (%)", fontsize=12)
plt.ylabel("")
plt.title(
    "Prévalence des comorbidités sélectionnées selon le statut vital",
    fontsize=14,
    weight="bold"
)

plt.grid(axis="x", linestyle="--", alpha=0.3)
plt.legend(title="Statut vital", frameon=False)

ax = plt.gca()
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_visible(False)

plt.xlim(0, pivot_comorb.to_numpy().max() * 1.18)

plt.tight_layout()

plt.savefig(
    "comorbidity_prevalence_by_death_status_fr.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
import polars as pl
import matplotlib.pyplot as plt
import numpy as np

# If df_patient is LazyFrame, collect it
if isinstance(df_patient, pl.LazyFrame):
    df_patient_plot = df_patient.collect()
else:
    df_patient_plot = df_patient

# Optional: English labels for the report
comorbidity_labels = {
    "Cardiopathie ischémique": "Ischemic heart disease",
    "Fibrillation atriale": "Atrial fibrillation",
    "Insuffisance cardiaque chronique": "Chronic heart failure",
    "Pacemaker": "Pacemaker",
    "Pontage aorto-coronarien": "Coronary artery bypass grafting",
    "Insuffisance rénale chronique": "Chronic kidney disease",
    "Antécédent d'AVC": "Previous stroke",
    "Cirrhose": "Cirrhosis",
    "Cancer actif": "Active cancer",
    "Immunodépression": "Immunosuppression",
    "Dyslipidémie": "Dyslipidemia",
    "Diabète": "Diabetes",
}

# Compute prevalence by death status
comorbidity_by_death = []

for c in comorbidity_cols:
    tmp = (
        df_patient_plot
        .group_by("death_status")
        .agg(
            (pl.col(c).sum() / pl.len() * 100).alias("prevalence_percent")
        )
        .with_columns(pl.lit(c).alias("comorbidity"))
    )
    comorbidity_by_death.append(tmp)

comorbidity_by_death = pl.concat(comorbidity_by_death)

# Convert to pandas for easier plotting
comorbidity_by_death_pd = comorbidity_by_death.to_pandas()

pivot_comorb = (
    comorbidity_by_death_pd
    .pivot(
        index="comorbidity",
        columns="death_status",
        values="prevalence_percent"
    )
    .fillna(0)
)

# Rename comorbidities for a cleaner figure
pivot_comorb.index = [
    comorbidity_labels.get(x, x) for x in pivot_comorb.index
]

# Keep only existing death-status columns in a stable order
death_order = [x for x in ["Alive", "Dead"] if x in pivot_comorb.columns]
pivot_comorb = pivot_comorb[death_order]

# Sort by prevalence among deceased patients if Dead exists, otherwise by first column
sort_col = "Dead" if "Dead" in pivot_comorb.columns else pivot_comorb.columns[0]
pivot_comorb = pivot_comorb.sort_values(by=sort_col)

# Plot
y_pos = np.arange(len(pivot_comorb.index))
bar_height = 0.38

plt.figure(figsize=(10, 7))

for i, status in enumerate(pivot_comorb.columns):
    values = pivot_comorb[status].values
    offset = (i - (len(pivot_comorb.columns) - 1) / 2) * bar_height

    bars = plt.barh(
        y_pos + offset,
        values,
        height=bar_height,
        label=status,
        edgecolor="white",
        linewidth=1.0,
        alpha=0.85
    )

    # Add percentage labels
    for bar, value in zip(bars, values):
        if value > 0:
            plt.text(
                value + 0.4,
                bar.get_y() + bar.get_height() / 2,
                f"{value:.1f}%",
                va="center",
                fontsize=9
            )

plt.yticks(y_pos, pivot_comorb.index, fontsize=10)
plt.xlabel("Prevalence among patients (%)", fontsize=12)
plt.ylabel("")
plt.title(
    "Prevalence of selected comorbidities according to death status",
    fontsize=14,
    weight="bold"
)

plt.grid(axis="x", linestyle="--", alpha=0.3)
plt.legend(title="Death status", frameon=False)

ax = plt.gca()
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_visible(False)

plt.xlim(0, pivot_comorb.to_numpy().max() * 1.18)

plt.tight_layout()

# Save for Overleaf
plt.savefig("comorbidity_prevalence_by_death_status.png", dpi=300, bbox_inches="tight")

plt.show()

In [ ]:
import polars as pl
import matplotlib.pyplot as plt
import numpy as np

# If df_patient is LazyFrame, collect it
if isinstance(df_patient, pl.LazyFrame):
    df_patient_plot = df_patient.collect()
else:
    df_patient_plot = df_patient

# Compute prevalence of each comorbidity
comorbidity_prevalence = (
    df_patient_plot
    .select([
        (pl.col(c).sum() / pl.len() * 100).alias(c)
        for c in comorbidity_cols
    ])
    .transpose(
        include_header=True,
        header_name="comorbidity",
        column_names=["prevalence_percent"]
    )
    .sort("prevalence_percent")
)

comorbidity_names = comorbidity_prevalence["comorbidity"].to_list()
prevalence_values = comorbidity_prevalence["prevalence_percent"].to_list()

y_pos = np.arange(len(comorbidity_names))

plt.figure(figsize=(9, 6.5))

bars = plt.barh(
    y_pos,
    prevalence_values,
    edgecolor="white",
    linewidth=1.1,
    alpha=0.85
)

# Add percentage labels at the end of each bar
for bar, value in zip(bars, prevalence_values):
    plt.text(
        value + 0.5,
        bar.get_y() + bar.get_height() / 2,
        f"{value:.1f}%",
        va="center",
        fontsize=10
    )

plt.yticks(y_pos, comorbidity_names, fontsize=10)
plt.xlabel("Prevalence among patients (%)", fontsize=12)
plt.ylabel("")
plt.title("Prevalence of selected comorbidities", fontsize=14, weight="bold")

plt.grid(axis="x", linestyle="--", alpha=0.3)

ax = plt.gca()
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_visible(False)

plt.xlim(0, max(prevalence_values) * 1.15)

plt.tight_layout()

# Save for Overleaf
plt.savefig("comorbidity_prevalence.png", dpi=300, bbox_inches="tight")

plt.show()

In [ ]:
import polars as pl
import matplotlib.pyplot as plt
import numpy as np

# Convert LazyFrame to DataFrame if needed
if isinstance(df_patient, pl.LazyFrame):
    df_patient_plot = df_patient.collect()
else:
    df_patient_plot = df_patient

# Standardize gender labels
df_patient_plot = df_patient_plot.with_columns(
    pl.when(pl.col("gender") == "F").then(pl.lit("Female"))
    .when(pl.col("gender") == "M").then(pl.lit("Male"))
    .otherwise(pl.col("gender"))
    .alias("gender_label")
)

# Build summary table
gender_death = (
    df_patient_plot
    .drop_nulls(["death_status", "gender_label"])
    .group_by(["death_status", "gender_label"])
    .agg(pl.len().alias("n"))
    .with_columns(
        (pl.col("n") / pl.col("n").sum().over("death_status") * 100).alias("percent")
    )
)

gender_pd = gender_death.to_pandas()

print(gender_pd)

# Stop if no data
if gender_pd.empty:
    raise ValueError("No data available for plotting. Check death_status and gender values.")

pivot_gender = (
    gender_pd
    .pivot(index="death_status", columns="gender_label", values="percent")
    .fillna(0)
)

print(pivot_gender)

# Use existing order only
preferred_death_order = ["Alive", "Dead"]
existing_death_order = [x for x in preferred_death_order if x in pivot_gender.index]

if len(existing_death_order) > 0:
    pivot_gender = pivot_gender.loc[existing_death_order]

preferred_gender_order = ["Female", "Male"]
existing_gender_order = [x for x in preferred_gender_order if x in pivot_gender.columns]

if len(existing_gender_order) > 0:
    pivot_gender = pivot_gender[existing_gender_order]

# Final safety check
if pivot_gender.empty or pivot_gender.shape[1] == 0:
    raise ValueError("The pivot table is empty after ordering. Check the actual labels in death_status and gender.")

x = np.arange(len(pivot_gender.index))
width = 0.35

plt.figure(figsize=(7, 4.8))

for i, gender in enumerate(pivot_gender.columns):
    values = pivot_gender[gender].values
    offset = (i - (len(pivot_gender.columns) - 1) / 2) * width

    bars = plt.bar(
        x + offset,
        values,
        width=width,
        label=gender,
        edgecolor="white",
        linewidth=1.2,
        alpha=0.85
    )

    for bar, value in zip(bars, values):
        plt.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 1,
            f"{value:.1f}%",
            ha="center",
            va="bottom",
            fontsize=10
        )

plt.xticks(x, pivot_gender.index, fontsize=11)
plt.xlabel("Death status", fontsize=12)
plt.ylabel("Patients (%)", fontsize=12)
plt.title("Gender distribution according to death status", fontsize=14, weight="bold")

plt.ylim(0, pivot_gender.to_numpy().max() * 1.20)
plt.grid(axis="y", linestyle="--", alpha=0.3)

plt.legend(title="Gender", frameon=False)

ax = plt.gca()
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()
plt.savefig("gender_by_death_status.png", dpi=300, bbox_inches="tight")
plt.show()